<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 13 · Regresión múltiple y diagnóstico

La semana pasada ajustaste una recta con una sola variable y aprendiste a leer el coeficiente en
dólares. Hoy entran varias a la vez y aparecen tres problemas que con una sola no existen. **El
coeficiente cambia de significado**, porque mide el efecto de una variable manteniendo las otras
quietas: hay una variable de Advertising que era significativa sola y deja de serlo acompañada. **Media
tabla no existe todavía el día que hay que decidir**, y meterla produce un R² de 0,99 que no es un
ajuste sino una identidad contable: esa es la parte cara de hoy. Y **los residuos hablan**: si sabes
mirarlos te dicen que el modelo está mal especificado antes de que lo descubra un gerente.

> **Hoy haces** · Ajustas la regresión múltiple de Advertising con `statsmodels` y traduces el
> «manteniendo todo lo demás constante» a una frase de negocio (90 min). Metes la ciudad como variable
> categórica y lees la categoría de referencia. Separas las variables **disponibles al momento de
> decidir** de las que solo se conocen cuando el mes ya cerró, y compruebas en números por qué las
> segundas producen un R² altísimo e inservible. Calculas el VIF de cada predictor y decides cuáles se
> quedan. Cierras con los cuatro gráficos de residuos y un veredicto escrito por supuesto.
>
> **Entrega** · Este cuaderno ejecutado, el modelo de pronóstico del caso del grupo construido **solo
> con variables disponibles al momento de decidir**, con la tabla de VIF y las descartadas
> justificadas, los cuatro diagnósticos con su veredicto, y **una frase accionable para el gerente** sin
> la palabra «coeficiente». Nombre de archivo: `lab_13_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
BASE_URL = ("https://raw.githubusercontent.com/mayait/"
            "CursoAnalisisDatos_IA_2026/main/sitio/datos")
ARCHIVOS = ["clientes.csv", "productos.csv", "sucursales.csv", "ventas.csv",
            "ventas_limpias.csv", "marketing_mensual.csv",
            "experimento_reactivacion.csv"]
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if (p / "ventas.csv").exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se descargan los siete archivos una vez.
    from urllib.request import urlretrieve
    DATOS = Path("datos")
    DATOS.mkdir(exist_ok=True)
    for archivo in ARCHIVOS:
        if not (DATOS / archivo).exists():
            urlretrieve(f"{BASE_URL}/{archivo}", DATOS / archivo)

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. La prensa que dejó de servir

El gerente de marketing reparte el presupuesto entre televisión, radio y prensa, así que la pregunta
nunca es «¿sirve la prensa?» sino «¿sirve la prensa **además** de lo que ya hago en televisión y
radio?». Son dos preguntas distintas y dan respuestas distintas. Ajustamos primero las tres regresiones
simples —una por medio, como la semana pasada— y después las tres variables juntas. Lo interesante es
la diferencia.

In [ ]:
import statsmodels.api as sm

URL_ADV = ("https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/"
           "master/data/Advertising.csv")
adv = pd.read_csv(URL_ADV, index_col=0)

filas = []
for medio in ["TV", "Radio", "Newspaper"]:
    s = sm.OLS(adv["Sales"], sm.add_constant(adv[[medio]])).fit()
    filas.append({"medio": medio, "coef. solo": s.params.iloc[1], "p-valor solo": s.pvalues.iloc[1],
                  "R² solo": s.rsquared})

multi = sm.OLS(adv["Sales"], sm.add_constant(adv[["TV", "Radio", "Newspaper"]])).fit()
comparacion = pd.DataFrame(filas).set_index("medio")
comparacion["coef. acompañado"] = multi.params[["TV", "Radio", "Newspaper"]]
comparacion["p-valor acompañado"] = multi.pvalues[["TV", "Radio", "Newspaper"]]

print(f"{len(adv)} mercados · inversión en miles de dólares · ventas en miles de unidades\n")
print(comparacion[["coef. solo", "p-valor solo", "R² solo",
                   "coef. acompañado", "p-valor acompañado"]]
      .to_string(float_format=lambda v: f"{v:,.6f}"))
print(f"\nR² del modelo con las tres juntas: {multi.rsquared:.4f} "
      f"(el mejor modelo simple llegaba a {comparacion['R² solo'].max():.4f})")
print(f"correlación entre Radio y Newspaper: {adv['Radio'].corr(adv['Newspaper']):.4f}  "
      "← la clave de lo que acaba de pasar")

📌 **La prensa vale 0,054693 unidades por cada mil dólares cuando se mira sola, con un p-valor de
0,001148, y vale −0,001037 con un p-valor de 0,859915 cuando se mira junto a las otras dos.** No cambió
el dato ni el método: cambió la pregunta.

La explicación está en la última línea: **radio y prensa correlacionan 0,3541.** Donde la empresa
invierte en radio también invierte en prensa, así que la prensa sola se lleva el crédito de las ventas
que produjo la radio. Con la radio dentro no queda nada que explicar y su coeficiente se desploma.

Por eso el coeficiente de una regresión múltiple se lee siempre con la coletilla completa:

> **Cada mil dólares más de televisión se asocian con 45,8 unidades más vendidas, *manteniendo
> constantes la inversión en radio y en prensa*.**

Sin esa segunda parte la frase es falsa. Con ella, la recomendación es directa: *«en el reparto actual
la prensa no aporta nada que la radio y la televisión no expliquen ya»*.

## 2. La ciudad no es un número: variables indicadoras

Comercial Andina factura en cinco ciudades y en el canal en línea. Si quieres meter la ciudad en un
modelo no puedes escribir «Quito = 1, Guayaquil = 2, Cuenca = 3»: eso le diría al modelo que Cuenca es
el triple de Quito y que la distancia Quito–Guayaquil es la misma que Guayaquil–Cuenca. Todo falso.

La solución es una columna de ceros y unos por categoría —una **variable indicadora**— dejando una
fuera. La que se deja fuera es la **categoría de referencia** y los demás coeficientes se leen contra
ella. Primero la tabla agregada: un mes y una ciudad por fila.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
sucursales = pd.read_csv(DATOS / "sucursales.csv")
marketing = pd.read_csv(DATOS / "marketing_mensual.csv", parse_dates=["mes"])

v = ventas.merge(sucursales[["sucursal_id", "ciudad"]], on="sucursal_id",
                 how="left", validate="m:1")
v["monto"] = v["cantidad"] * v["precio_unitario"] * (1 - v["descuento"])
v["mes"] = v["fecha"].values.astype("datetime64[M]")

agg = (v.groupby(["mes", "ciudad"])
       .agg(ventas=("monto", "sum"), unidades=("cantidad", "sum"),
            facturas=("factura_id", "nunique"), clientes=("cliente_id", "nunique"))
       .reset_index()
       .merge(marketing[["mes", "inversion_volantes", "inversion_radio", "inversion_digital"]],
              on="mes", how="left"))

# Julio de 2026 no está cerrado: solo tiene notas de crédito (misma decisión que en el lab 12).
agg = agg[agg["mes"] < "2026-07-01"].copy()
agg["indice_mes"] = (agg["mes"].dt.year - 2024) * 12 + agg["mes"].dt.month - 1

print(f"{len(agg)} filas = {agg['mes'].nunique()} meses × {agg['ciudad'].nunique()} ciudades")
print(f"ventas medias por mes y ciudad: {agg['ventas'].mean():,.2f}\n")
print(agg.groupby("ciudad")["ventas"].agg(["size", "mean", "sum"])
      .sort_values("mean", ascending=False).to_string(float_format=lambda x: f"{x:,.2f}"))

In [ ]:
# ✅ Comprobación 1 · la tabla agregada es la base de todo el cuaderno
assert len(agg) == 180, f"agg debería tener 180 filas (30 meses × 6 ciudades) y tiene {len(agg)}"
assert abs(agg["ventas"].mean() - 15596.82) < 1.0, \
    "La venta media por mes y ciudad no coincide: ¿filtraste julio de 2026 y usaste ventas_limpias.csv?"
assert agg["ventas"].isna().sum() == 0 and agg["inversion_radio"].isna().sum() == 0, \
    "Hay meses sin inversión de marketing: revisa el merge con marketing_mensual.csv"

print("Comprobación 1 superada ✓  180 filas, sin huecos, media 15 596,82")

In [ ]:
indicadoras = pd.get_dummies(agg["ciudad"], prefix="ciudad", drop_first=True, dtype=float)
print("columnas creadas por get_dummies con drop_first=True:")
print(" ", list(indicadoras.columns))
print(f"\ncategoría de referencia (la que NO tiene columna): "
      f"{sorted(agg['ciudad'].unique())[0]}\n")
print(pd.concat([agg[["mes", "ciudad"]], indicadoras], axis=1).head(7).to_string(index=False))

X_ciudad = pd.concat([agg[["inversion_volantes", "indice_mes"]], indicadoras], axis=1)
modelo_ciudad = sm.OLS(agg["ventas"], sm.add_constant(X_ciudad)).fit()

print(modelo_ciudad.summary().tables[1])
print(f"\nR² {modelo_ciudad.rsquared:.4f} · R² ajustado {modelo_ciudad.rsquared_adj:.4f}")

media_cuenca = agg.loc[agg["ciudad"] == "Cuenca", "ventas"].mean()
media_quito = agg.loc[agg["ciudad"] == "Quito", "ventas"].mean()
print(f"\ncomprobación: media mensual de Cuenca {media_cuenca:,.2f} · "
      f"de Quito {media_quito:,.2f} · diferencia {media_quito - media_cuenca:,.2f}")
print(f"coeficiente ciudad_Quito del modelo: {modelo_ciudad.params['ciudad_Quito']:,.2f}")

La tabla se lee así, y cada línea es una frase que un gerente entiende:

- **El intercepto, 4 643,04, es Cuenca con inversión cero y en el mes cero.** La referencia no
  desaparece: se esconde en el intercepto.
- **`ciudad_Quito` = 13 204,49**: Quito factura eso al mes más que Cuenca, con la misma inversión en
  volantes y en el mismo mes. **`ciudad_Loja` = −8 168,05** es la otra cara.
- **`inversion_volantes` = 5,33** son cinco dólares con treinta y tres de venta por dólar de volantes
  **dentro de cada ciudad**, no mezclando ciudades.

Fíjate en la comprobación: la diferencia bruta de medias entre Quito y Cuenca es 13 204,49 y el
coeficiente es **el mismo número**, porque las seis ciudades tienen los mismos treinta meses. **En una
tabla desbalanceada —el caso normal— los dos números se separan, y ahí el bueno es el coeficiente**,
que compara ciudades con la misma inversión.

## 3. Fuga de información: predictores que no existen el día de decidir

El gerente de Loja no pregunta cuánto facturó: pregunta **cuánto va a facturar**, y lo pregunta el día
25 del mes anterior para programar inventario y personal. Esa fecha —**el momento en que hay que
decidir**— manda sobre qué columnas pueden entrar al modelo. En `agg` hay dos clases:

| clase | columnas | ¿se conoce el día 25? |
|---|---|---|
| **Disponible al decidir** | `inversion_radio`, `inversion_digital`, `inversion_volantes`, `indice_mes`, la ciudad | **Sí.** El presupuesto ya está comprometido y el calendario no depende de nadie |
| **Conocida después** | `unidades`, `facturas`, `clientes` | **No.** Se sabe cuando el mes cerró, al mismo tiempo que `ventas` |

Usar una columna de la segunda clase como predictora es **fuga de información** (*data leakage*): el
modelo aprende de un dato que no va a existir el día que haya que usarlo. Es la misma disciplina de las
dos ventanas que formalizas la semana que viene con el abandono, aplicada aquí a un mes:

```
|------- ventana de observación -------|-------- ventana de resultado --------|
  de aquí salen TODAS las predictoras     aquí viven unidades, facturas,
  (hasta el día 25)                       clientes… y ventas, lo que se pronostica
```

`unidades`, `facturas` y `clientes` no son predictores de `ventas`: son **consecuencias mecánicas** de
las ventas del mismo mes. Un mes con más ventas tiene más unidades y más facturas por definición
contable, no por causalidad. Mide cuánto de mecánico tiene esa relación.

In [ ]:
CONSECUENCIAS = ["unidades", "facturas", "clientes"]
identidad = pd.DataFrame({
    "variable": CONSECUENCIAS,
    "correlación con ventas": [agg["ventas"].corr(agg[c]) for c in CONSECUENCIAS],
    "R² de ventas ~ esa variable sola": [
        sm.OLS(agg["ventas"], sm.add_constant(agg[[c]])).fit().rsquared for c in CONSECUENCIAS],
    "¿se conoce el día 25?": ["no", "no", "no"],
})
print(identidad.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

PRECIO_MEDIO = agg["ventas"].sum() / agg["unidades"].sum()
error_identidad = (agg["ventas"] - agg["unidades"] * PRECIO_MEDIO).abs()
print(f"\nprecio medio implícito de una unidad: {PRECIO_MEDIO:.4f} dólares")
print(f"'pronóstico' ventas = unidades × {PRECIO_MEDIO:.2f} → error absoluto medio "
      f"{error_identidad.mean():,.2f} sobre una venta media de {agg['ventas'].mean():,.2f} "
      f"({(error_identidad / agg['ventas']).mean():.2%})")
print("\nNo hace falta ajustar nada: multiplicar unidades por el precio medio ya reconstruye la venta.")

📌 **`unidades` sola explica el 99,23 % de la variación de las ventas, y multiplicarla por 2,19 dólares
reconstruye la facturación con un 4,4 % de error sin ajustar ningún modelo.** Eso no es un hallazgo: es
la definición de facturación. Un R² de 0,99 obtenido así **no mide capacidad de pronóstico, mide una
identidad contable**, y el día 25 no hay ni una sola de esas tres columnas sobre la mesa.

Regla que se aplica antes de escribir `sm.OLS`: **de cada predictor, pregunta en qué momento se conoce
su valor.** Si la respuesta es «al mismo tiempo que la variable objetivo o después», no es un predictor.

Y hay un segundo problema, independiente del primero: las tres se pisan entre ellas.

### Multicolinealidad y el factor VIF

Cuando dos predictores se mueven juntos, mínimos cuadrados no puede repartir el crédito y **reparte al
azar**: coeficiente enorme a uno y otro enorme de signo contrario al otro, que se cancelan. El ajuste
queda perfecto y los coeficientes, inservibles. El **factor de inflación de la varianza (VIF)** lo mide:
se regresa cada predictor contra todos los demás y se mira cuánto lo explican.

> VIF = 1 / (1 − R² de esa regresión auxiliar)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

TODAS_LAS_VARIABLES = ["inversion_volantes", "inversion_radio", "inversion_digital",
                       "indice_mes", "unidades", "facturas", "clientes"]
X_todo = pd.concat([agg[TODAS_LAS_VARIABLES], indicadoras], axis=1)
modelo_todo = sm.OLS(agg["ventas"], sm.add_constant(X_todo)).fit()

Xv = sm.add_constant(X_todo).astype(float)
vif = pd.DataFrame({
    "variable": Xv.columns,
    "VIF": [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])],
}).query("variable != 'const'")


def tramo(v):
    if v < 5:
        return "1 – 5   · sin problema"
    if v < 10:
        return "5 – 10  · vigilar"
    if v < 30:
        return "10 – 30 · grave: no interpretes el coeficiente"
    return ">30     · redundante: sobra una variable"


vif["interpretación"] = vif["VIF"].apply(tramo)
vif["R² auxiliar"] = 1 - 1 / vif["VIF"]

print(f"R² del modelo con las {X_todo.shape[1]} variables: {modelo_todo.rsquared:.6f} "
      f"(ajustado {modelo_todo.rsquared_adj:.6f})\n")
print(vif.sort_values("VIF", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

| VIF | qué significa | qué hacer |
|---|---|---|
| **1 – 5** | El predictor aporta información propia | Nada. Déjalo |
| **5 – 10** | Comparte bastante con otro | Vigílalo. Mira si el coeficiente es estable |
| **10 – 30** | El 90 % o más de su variación la explican los demás | **No interpretes su coeficiente** |
| **> 30** | Es prácticamente una copia | Sobra. Elimínala o combínala con la otra |

El veredicto es inmediato: **`facturas` tiene VIF 116,61 y `clientes` 115,96.** Sus regresiones
auxiliares llegan al 99,1 % de R²: saber cuántos clientes compraron te dice cuántas facturas hubo.

Un matiz que hay que decir en voz alta: **el VIF alto de una indicadora casi nunca es un problema.**
`ciudad_Nacional` sale alto porque el canal en línea tiene otro perfil, no porque sobre. Las indicadoras
de un mismo grupo se evalúan juntas o no se evalúan.

Ahora la prueba de fuego: si el VIF alto es un problema de verdad, se ve en la estabilidad de los
coeficientes.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

POCAS = ["inversion_volantes", "indice_mes", "unidades"]
X_pocas = pd.concat([agg[POCAS], indicadoras], axis=1)

rng2 = np.random.default_rng(SEED)
coef_todo, coef_pocas = [], []
for _ in range(6):
    mitad = rng2.permutation(len(agg))[:len(agg) // 2]
    coef_todo.append(sm.OLS(agg["ventas"].iloc[mitad],
                            sm.add_constant(X_todo.iloc[mitad])).fit().params)
    coef_pocas.append(sm.OLS(agg["ventas"].iloc[mitad],
                             sm.add_constant(X_pocas.iloc[mitad])).fit().params)
T, P = pd.DataFrame(coef_todo), pd.DataFrame(coef_pocas)

print("Coeficientes del modelo CON TODO, ajustado seis veces sobre mitades aleatorias:\n")
print(T[["facturas", "clientes", "unidades", "inversion_volantes"]]
      .to_string(float_format=lambda v: f"{v:,.2f}"))
print("\nCoeficientes del modelo REDUCIDO (una sola de las tres consecuencias) sobre las mismas seis mitades:\n")
print(P[POCAS].to_string(float_format=lambda v: f"{v:,.2f}"))

print(f"\ninestabilidad (desviación / media) de 'facturas' con todo : "
      f"{T['facturas'].std() / abs(T['facturas'].mean()):.3f}")
print(f"inestabilidad de 'unidades' en el modelo reducido    : "
      f"{P['unidades'].std() / abs(P['unidades'].mean()):.3f}")
print(f"\nR² validación cruzada · modelo con todo ({X_todo.shape[1]} variables) : "
      f"{cross_val_score(LinearRegression(), X_todo, agg['ventas'], cv=kf, scoring='r2').mean():.4f}")
print(f"R² validación cruzada · modelo reducido ({X_pocas.shape[1]} variables) : "
      f"{cross_val_score(LinearRegression(), X_pocas, agg['ventas'], cv=kf, scoring='r2').mean():.4f}")

📌 **El coeficiente de `facturas` va de −16,28 a +8,18 según qué mitad de los datos le toque.** Cambia
de signo: con una mitad el informe diría «cada factura resta 16 dólares de venta» y con otra diría
«suma 8». Lo mismo, al revés, con `clientes`: los dos se compensan porque miden lo mismo.

`unidades`, en cambio, vale entre 2,08 y 2,24 en las seis mitades: una variación del **1,9 %**. Y ese
número ya lo viste: **2,19 es el precio medio de una unidad.** Su coeficiente es estable porque no
estima nada, devuelve la identidad contable. Estabilidad y utilidad no son lo mismo.

Ninguno de los dos modelos de esta tabla sirve para pronosticar: los dos usan columnas del mes cerrado.

### El modelo que sí se puede correr el día 25

Solo con lo que está sobre la mesa antes de que empiece el mes: las tres inversiones publicitarias y el
índice de mes. Y una segunda versión que añade la ciudad, que también se conoce de antemano —una tienda
no cambia de ciudad— y que en la sección 2 ya demostró valer mucho.

In [ ]:
PREVIAS = ["inversion_radio", "inversion_digital", "inversion_volantes", "indice_mes"]
X_previo = agg[PREVIAS]
X_previo_ciudad = pd.concat([agg[PREVIAS], indicadoras], axis=1)

modelo_previo = sm.OLS(agg["ventas"], sm.add_constant(X_previo)).fit()
modelo_previo_ciudad = sm.OLS(agg["ventas"], sm.add_constant(X_previo_ciudad)).fit()

comparacion_fuga = pd.DataFrame({
    "predictores": [X_todo.shape[1], X_previo.shape[1], X_previo_ciudad.shape[1]],
    "R² de entrenamiento": [modelo_todo.rsquared, modelo_previo.rsquared,
                            modelo_previo_ciudad.rsquared],
    "R² de validación cruzada": [
        cross_val_score(LinearRegression(), X, agg["ventas"], cv=kf, scoring="r2").mean()
        for X in [X_todo, X_previo, X_previo_ciudad]],
    "¿se puede correr el día 25?": ["NO · usa el mes cerrado", "sí", "sí"],
}, index=["con fuga (7 variables + ciudad)", "solo inversión y mes",
          "inversión, mes y ciudad"])

print(comparacion_fuga.to_string(float_format=lambda v: f"{v:,.4f}"))
print()
print(modelo_previo_ciudad.summary().tables[1])
print(f"\nEl único predictor de inversión con p-valor por debajo de 0,05 es "
      f"inversion_volantes ({modelo_previo_ciudad.pvalues['inversion_volantes']:.4f}).")

In [ ]:
# ✅ Comprobación 2 · la fuga de información, en dos asserts
assert modelo_todo.rsquared > 0.99, \
    "El modelo con fuga debería dar R² > 0,99: revisa que X_todo incluya unidades, facturas y clientes"
assert modelo_previo.rsquared < 0.20, \
    "El modelo sin fuga debería quedarse por debajo de 0,20 de R²: ¿se te coló alguna consecuencia?"
assert abs(modelo_previo_ciudad.rsquared - 0.8507) < 0.01, \
    "El modelo con ciudad no coincide: revisa que uses drop_first=True en las indicadoras"
assert not set(PREVIAS) & set(CONSECUENCIAS), \
    "Hay una variable de la ventana de resultado dentro de PREVIAS: quítala"

print("Comprobación 2 superada ✓  0,99 con fuga y 0,15 sin ella: esa es la diferencia")

📌 **El R² honesto de lo que se puede pronosticar el día 25 es 0,15, no 0,99.** Con la ciudad dentro
sube a 0,85, y conviene entender el salto: casi todo es el **nivel estructural de cada plaza** —Quito
factura más que Loja todos los meses— y no el efecto de la publicidad. El único medio que mueve la
aguja es el volanteo; radio y digital no son distinguibles de cero.

Esa es la frase que va al informe, y es mucho menos vistosa que un 0,99:

> *«Sabiendo la ciudad, el mes y el presupuesto publicitario comprometido, el modelo explica el 85 % de
> la variación de la facturación mensual. La mayor parte de ese 85 % es el tamaño de cada plaza; el
> único medio con efecto medible es el volanteo, con 5,33 dólares de venta por dólar invertido.»*

**La caída de 0,99 a 0,85 no es un modelo peor: es el primer modelo verdadero.**

## 4. Los cuatro supuestos y sus cuatro gráficos

Mínimos cuadrados funciona bien si se cumplen cuatro supuestos. Cada uno tiene su gráfico y su prueba
numérica, y cada uno se declara con un veredicto por escrito.

| supuesto | qué dice | gráfico | prueba |
|---|---|---|---|
| **Linealidad** | La relación es una recta (o un plano) | Residuos contra valores ajustados | RESET de Ramsey |
| **Independencia** | El error de una observación no dice nada del de la siguiente | Residuos en el orden de los datos | Durbin–Watson |
| **Normalidad** | Los residuos siguen una campana | Gráfico cuantil-cuantil | Jarque–Bera |
| **Homocedasticidad** | El error tiene el mismo tamaño en todo el rango | Raíz del residuo estandarizado contra ajustados | White y Breusch–Pagan |

Los aplicamos al modelo de Advertising con las tres variables, que es el que llevaríamos a la reunión.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from scipy import stats

resid = multi.resid
ajust = multi.fittedvalues
estand = resid / resid.std()

fig, axes = plt.subplots(2, 2, figsize=(12.5, 8))

axes[0, 0].scatter(ajust, resid, s=16, alpha=0.6, color="#4C72B0")
suave = pd.DataFrame({"a": ajust, "r": resid}).sort_values("a")
axes[0, 0].plot(suave["a"], suave["r"].rolling(25, center=True, min_periods=5).mean(),
                color="#C44E52", linewidth=2)
axes[0, 0].axhline(0, color="grey", linewidth=1)
axes[0, 0].set_xlabel("valores ajustados")
axes[0, 0].set_ylabel("residuo")
axes[0, 0].set_title("1 · Linealidad: la media de los residuos dibuja una U", fontsize=11)

axes[0, 1].plot(np.arange(len(resid)), resid, marker="o", markersize=3,
                linewidth=0.6, color="#55A868")
axes[0, 1].axhline(0, color="grey", linewidth=1)
axes[0, 1].set_xlabel("orden de la fila en el archivo")
axes[0, 1].set_ylabel("residuo")
axes[0, 1].set_title("2 · Independencia: ruido sin patrón", fontsize=11)

stats.probplot(resid, dist="norm", plot=axes[1, 0])
axes[1, 0].get_lines()[0].set(marker="o", markersize=4, color="#4C72B0", alpha=0.7)
axes[1, 0].get_lines()[1].set(color="#C44E52", linewidth=2)
axes[1, 0].set_title("3 · Normalidad: la cola izquierda se despega", fontsize=11)
axes[1, 0].set_xlabel("cuantiles teóricos")
axes[1, 0].set_ylabel("cuantiles observados")

axes[1, 1].scatter(ajust, np.sqrt(np.abs(estand)), s=16, alpha=0.6, color="#DD8452")
s2 = pd.DataFrame({"a": ajust, "r": np.sqrt(np.abs(estand))}).sort_values("a")
axes[1, 1].plot(s2["a"], s2["r"].rolling(25, center=True, min_periods=5).mean(),
                color="#C44E52", linewidth=2)
axes[1, 1].set_xlabel("valores ajustados")
axes[1, 1].set_ylabel("√|residuo estandarizado|")
axes[1, 1].set_title("4 · Homocedasticidad: el error se dispara en los extremos", fontsize=11)

fig.suptitle("Dos supuestos se cumplen y dos no: el modelo está mal especificado",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
reset = linear_reset(multi, power=2, use_f=True)
jb_stat, jb_p, sesgo, curtosis = jarque_bera(resid)
dw = durbin_watson(resid)
bp_p = het_breuschpagan(resid, multi.model.exog)[1]
white_p = het_white(resid, multi.model.exog)[1]

diag = pd.DataFrame([
    ("Linealidad", "RESET de Ramsey", reset.pvalue,
     "NO SE CUMPLE · falta un término en el modelo"),
    ("Independencia", f"Durbin-Watson = {dw:.4f}", np.nan,
     "SE CUMPLE · entre 1,5 y 2,5, sin autocorrelación"),
    ("Normalidad", "Jarque-Bera", jb_p,
     f"NO SE CUMPLE · asimetría {sesgo:.2f} y curtosis {curtosis:.2f}"),
    ("Homocedasticidad", "White", white_p,
     f"NO SE CUMPLE · Breusch-Pagan no lo ve (p = {bp_p:.4f}) porque solo busca patrones lineales"),
], columns=["supuesto", "prueba", "p-valor", "veredicto"])

print(diag.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"\nRegla: p-valor < 0,05 → se rechaza que el supuesto se cumpla.")

### El veredicto, y qué se hace con él

**Linealidad: no se cumple** (RESET, p = 1,01e-12). Los residuos dibujan una U: el modelo se queda
corto en los mercados de inversión baja y alta, y se pasa en el medio. Es la firma de una variable que
falta.

**Independencia: se cumple** (Durbin–Watson 2,0836, referencia 2). Aquí el orden de las filas es
arbitrario y la prueba tiene poco sentido; **en tus datos mensuales sí lo tiene**.

**Normalidad: no se cumple** (Jarque–Bera, p = 1,44e-33). Con 200 observaciones el teorema central del
límite salva los intervalos de confianza de los coeficientes: esto **no invalida el modelo**, invalida
cualquier intervalo de predicción para un mercado concreto.

**Homocedasticidad: no se cumple** (White, p = 4,62e-11). Breusch–Pagan da p = 0,1623 y no la detecta,
porque solo busca heterocedasticidad lineal y aquí crece en forma de U. **Una prueba que no rechaza no
es una prueba que confirma.**

Diagnosticar **no es aprobar o suspender el modelo: es saber para qué se puede usar y para qué no.** El
diagnóstico dice que falta un término y la teoría del negocio dice cuál —la televisión rinde más donde
ya hay radio—; ese arreglo está en el apéndice.

### 🌶️ Ejercicio 1 — Guiado

Ajusta la regresión múltiple de las ventas mensuales por ciudad **solo con variables disponibles el día
25**: las tres de inversión, `indice_mes` y las indicadoras de ciudad. Entrega tres cosas: (a) la tabla
de VIF con su tramo, (b) la frase de negocio del coeficiente de `inversion_volantes` con la coletilla
de «manteniendo constantes…» completa, y (c) el coeficiente de la ciudad que más factura leído contra
la referencia, comprobado con la diferencia bruta de medias.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: X_previo_ciudad ya está construido en la sección 3. Reutilízalo con sm.OLS y add_constant
# Pista 2: para el VIF necesitas sm.add_constant(...).astype(float) — variance_inflation_factor
#          falla con columnas booleanas o de tipo object
# Pista 3: la comprobación de (c) es agg.groupby("ciudad")["ventas"].mean() y restar la referencia.
#          Si el coeficiente y la diferencia bruta no se parecen, explica por qué: el modelo controla
#          por inversión y por mes, la media bruta no

### 🔥 Desafío

**Cambia la categoría de referencia y demuestra que el modelo es el mismo.** Vuelve a ajustar el modelo
de la sección 2 dejando fuera a Quito en lugar de a Cuenca —`pd.get_dummies(...).drop(columns=[...])`—
y comprueba que: (a) el R² es idéntico hasta el último decimal, (b) los valores ajustados son idénticos,
y (c) todos los coeficientes cambian pero las **diferencias entre ciudades** se conservan. Después
contesta la pregunta que importa: si los dos modelos son el mismo, ¿por qué elegir una referencia y no
otra? Escribe el criterio en dos líneas y aplícalo al caso de tu grupo.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: pd.get_dummies(agg["ciudad"], prefix="ciudad", drop_first=False, dtype=float)
#          y luego .drop(columns=["ciudad_Quito"])
# Pista 2: compara np.allclose(modelo_a.fittedvalues, modelo_b.fittedvalues)
# Pista 3: el criterio tiene que ver con quién lee la tabla. La referencia buena es la que hace que
#          los coeficientes se lean solos: la categoría más grande, la más antigua o el statu quo

### 🎯 Reto en clase (15 min)

En equipos y contra reloj: **el juicio a la variable**. Cada equipo recibe el modelo con las siete
variables de la sección 3 y tiene que **eliminar exactamente tres**, defendiendo cada eliminación con un
número: el VIF, el p-valor, el momento en que se conoce el dato o la caída del R² de validación
cruzada. Al final cada equipo reporta el R² de validación cruzada de su modelo **y si se puede correr el
día 25**. Un equipo que llegue a 0,99 y no pueda ejecutar su modelo antes de que cierre el mes pierde el
reto entero, por muy alto que sea el número.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: X_todo tiene las 7 + las 5 indicadoras. Elimina de las 7, las indicadoras van juntas
# Pista 2: cross_val_score(LinearRegression(), X_candidato, agg["ventas"], cv=kf, scoring="r2").mean()
# Pista 3: apunta el R² de validación cruzada ANTES de eliminar (0,9908) y el del modelo sin fuga
#          (0,8245) para saber contra qué comparas. Bajar de uno a otro no es empeorar: es dejar de
#          hacer trampa

## La trampa de hoy

⚠️ **Meter todas las variables disponibles porque el R² sube.** Es la trampa más común y la más difícil
de discutir en una reunión, porque el que la comete tiene el número más alto de la sala. Y tiene dos
capas: las variables se pisan entre ellas —y los coeficientes dejan de significar nada— y algunas ni
siquiera existen el día que hay que decidir.

Los dos modelos, lado a lado, con las cuatro cifras que deciden.

In [ ]:
resumen_trampa = pd.DataFrame({
    "modelo con TODO (12 variables)": [
        X_todo.shape[1],
        modelo_todo.rsquared,
        cross_val_score(LinearRegression(), X_todo, agg["ventas"], cv=kf, scoring="r2").mean(),
        T["facturas"].std() / abs(T["facturas"].mean()),
    ],
    "modelo SIN FUGA (9 variables)": [
        X_previo_ciudad.shape[1],
        modelo_previo_ciudad.rsquared,
        cross_val_score(LinearRegression(), X_previo_ciudad, agg["ventas"], cv=kf,
                        scoring="r2").mean(),
        np.nan,
    ],
}, index=["número de predictores", "R² de entrenamiento ← el que se presenta",
          "R² de validación cruzada", "inestabilidad del coeficiente en disputa (desv/media)"])

print(resumen_trampa.to_string(float_format=lambda v: f"{v:,.4f}"))
print(f"\n¿Se puede correr el día 25 del mes anterior?")
print(f"  modelo con TODO      : NO — unidades, facturas y clientes se conocen cuando el mes cerró")
print(f"  modelo SIN FUGA      : SÍ — inversión comprometida, calendario y ciudad")
print(f"\nGana el R² de entrenamiento : el modelo con todo, "
      f"por {modelo_todo.rsquared - modelo_previo_ciudad.rsquared:.4f}")
print(f"Gana el pronóstico          : el único que se puede ejecutar antes de conocer la respuesta")
print(f"\nEl coeficiente de 'facturas' del modelo grande va de {T['facturas'].min():,.2f} "
      f"a {T['facturas'].max():,.2f} según qué mitad de los datos le toque.")

📌 **0,9928 contra 0,8507, y el que gana no se puede ejecutar.** El modelo con fuga gana por catorce
puntos de R² donde no importa, entrega un coeficiente que va de −16,28 a +8,18 según la mitad de los
datos, y el día que el gerente lo necesita le falta la mitad de las columnas.

Nadie va a discutir tu R² en una reunión: van a discutir tu coeficiente, porque es el que dice qué
hacer el lunes. **Un modelo cuyo coeficiente cambia de signo con la mitad de los datos no tiene nada que
decir el lunes, y uno que necesita el mes cerrado no llega a tiempo a ninguna decisión.**

El orden de trabajo, siempre el mismo:

1. Escribe **cuándo hay que decidir** y descarta toda columna que no se conozca antes de esa fecha.
2. Ajusta con las variables que **el negocio** justifica, no con todas las que quedan.
3. Calcula el VIF. Lo que pase de 10 se revisa; de dos redundantes se queda la que se explique en una
   frase.
4. Diagnostica los residuos y escribe el veredicto de los cuatro supuestos.

## Entregable

Sube `lab_13_apellido.ipynb` con:

- La comparación de Advertising **solo contra acompañado**, con el caso de la prensa: 0,0547 con
  p = 0,0011 sola y −0,0010 con p = 0,8599 acompañada, y la correlación de 0,3541 como explicación.
- El modelo con la ciudad como indicadora, con la **categoría de referencia declarada** y un
  coeficiente comprobado contra la diferencia bruta de medias.
- La **clasificación de las variables del caso del grupo en dos listas** —disponibles al decidir y
  conocidas después— con la fecha de la decisión escrita. Es la parte que más se califica.
- La demostración de la fuga: 0,9928 de R² con las consecuencias dentro contra 0,8507 sin ellas, y la
  identidad `ventas ≈ unidades × 2,19` que explica de dónde salía el 0,99.
- La **tabla de VIF con su tramo** y qué variable se elimina y por qué: `facturas` 116,61 y `clientes`
  115,96.
- Los **cuatro gráficos de diagnóstico** con el veredicto de cada supuesto y la frase de para qué sirve
  el modelo y para qué no.
- Una fila en la bitácora de prompts: pídele al asistente el modelo con todas las variables y oblígalo a
  justificar **en qué momento se conoce cada predictor**. Deja cuáles no sobrevivieron a esa pregunta.

## Para tu equipo

- Antes de ajustar nada, escriban **la fecha en que la empresa toma la decisión** y tachen toda columna
  que no exista ese día. Es el filtro que más modelos mata y el que menos se aplica.
- El modelo se entrega con **la tabla de VIF y la lista de descartadas**. Descartar con un número al
  lado es un hallazgo; descartar sin número es una opinión.
- Si el R² honesto es bajo, esa es la noticia y se reporta. Un 0,85 que se puede ejecutar vale más que
  un 0,99 que necesita saber la respuesta para calcularla.
- La frase para el gerente no lleva «coeficiente» ni «significativo»: lleva una cantidad, una unidad y
  una decisión.

## Apéndice · para profundizar fuera de clase

Lo que sigue **no compite por los noventa minutos de clase**: es opcional y está aquí para quien quiera
llegar más lejos, o para consultarlo cuando el proyecto lo pida. Se ejecuta igual que el resto del
cuaderno, después de haber corrido todas las celdas anteriores.

### A1 · La tabla completa de `statsmodels` y la lectura en dinero

El `summary()` es la salida canónica de `statsmodels`. Estas son las dos tablas que hay que saber leer y la traducción de cada coeficiente a unidades de negocio.

In [ ]:
print(multi.summary().tables[1])
print(f"\nR² {multi.rsquared:.6f}  ·  R² ajustado {multi.rsquared_adj:.6f}  ·  "
      f"F {multi.fvalue:,.2f} (p = {multi.f_pvalue:.3g})")
print(f"\nLectura en dinero, con la coletilla obligatoria:")
for medio in ["TV", "Radio", "Newspaper"]:
    c = multi.params[medio]
    otros = [m for m in ["TV", "Radio", "Newspaper"] if m != medio]
    print(f"  · 1 000 USD más en {medio:10s} → {c * 1000:+7.1f} unidades, "
          f"manteniendo {otros[0]} y {otros[1]} constantes")

### A2 · La trampa de las indicadoras: `drop_first=False`

Si creas una columna por cada ciudad y las metes todas, la suma de las seis columnas es siempre 1, que
es exactamente la columna del intercepto: el modelo tiene infinitas soluciones equivalentes.
`statsmodels` no siempre falla, pero devuelve coeficientes sin sentido. Compruébalo.

In [ ]:
todas = pd.get_dummies(agg["ciudad"], prefix="ciudad", drop_first=False, dtype=float)
X_mal = pd.concat([agg[["inversion_volantes", "indice_mes"]], todas], axis=1)
malo = sm.OLS(agg["ventas"], sm.add_constant(X_mal)).fit()

print(f"suma de las {todas.shape[1]} columnas indicadoras, fila a fila: "
      f"{sorted(todas.sum(axis=1).unique())} ← es la columna del intercepto")
print(f"\nrango de la matriz: {np.linalg.matrix_rank(sm.add_constant(X_mal).values)} "
      f"de {X_mal.shape[1] + 1} columnas → hay una columna redundante\n")
print(malo.summary().tables[1])
print(f"\nR² del modelo mal construido : {malo.rsquared:.6f}")
print(f"R² del modelo bien construido: {modelo_ciudad.rsquared:.6f}  ← el mismo ajuste, "
      "otros coeficientes")

### A3 · El R² siempre sube. Siempre.

Añadir una variable a una regresión **nunca** puede bajar el R² de entrenamiento. En el peor de los
casos el método le asigna coeficiente cero y el ajuste queda igual; con datos reales siempre encuentra
alguna coincidencia que aprovechar. Por eso «subió el R²» no es un argumento para nada.

La demostración es brutal si las variables que añades no significan nada. Vamos a meterle al modelo de
Advertising cuatro columnas de números aleatorios, una por una.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold

rng = np.random.default_rng(SEED)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

X_ruido = adv[["TV", "Radio"]].copy()
etiquetas, filas = ["TV + Radio (variables de verdad)"], []
for k in range(5):
    if k > 0:
        X_ruido[f"ruido_{k}"] = rng.normal(size=len(adv))
        etiquetas.append(f"+ {k} columna(s) de ruido puro")
    ajuste = sm.OLS(adv["Sales"], sm.add_constant(X_ruido)).fit()
    r2_cv = cross_val_score(LinearRegression(), X_ruido, adv["Sales"], cv=kf, scoring="r2").mean()
    filas.append({"predictores": X_ruido.shape[1], "R² entrenamiento": ajuste.rsquared,
                  "R² ajustado": ajuste.rsquared_adj, "R² validación cruzada": r2_cv})

tabla_ruido = pd.DataFrame(filas, index=etiquetas)
print(tabla_ruido.to_string(float_format=lambda v: f"{v:.6f}"))
print(f"\nR² entrenamiento: {tabla_ruido['R² entrenamiento'].iloc[0]:.6f} → "
      f"{tabla_ruido['R² entrenamiento'].iloc[-1]:.6f}  (SUBE con basura)")
print(f"R² ajustado     : {tabla_ruido['R² ajustado'].iloc[0]:.6f} → "
      f"{tabla_ruido['R² ajustado'].iloc[-1]:.6f}  (BAJA)")
print(f"R² validación   : {tabla_ruido['R² validación cruzada'].iloc[0]:.6f} → "
      f"{tabla_ruido['R² validación cruzada'].iloc[-1]:.6f}  (BAJA)")

📌 **Cuatro columnas de ruido generado con `rng.normal()` suben el R² de entrenamiento de 0,897194 a
0,898440.** Nada de lo que hay en esas columnas tiene relación con las ventas: son números aleatorios.
El R² sube igual, porque el R² de entrenamiento mide lo bien que el modelo se acomodó a los datos que
ya vio, no lo bien que va a funcionar.

Las otras dos columnas de la tabla sí se dan cuenta:

- El **R² ajustado** cobra peaje por cada variable añadida —`1 − (1−R²)·(n−1)/(n−k−1)`— y baja de
  0,896151 a 0,895283.
- La **validación cruzada** baja de 0,884355 a 0,882879 porque mide sobre datos que el modelo no vio.

La regla, sin excepción: **el R² de entrenamiento no se usa nunca para comparar modelos con distinto
número de variables.** Se usa el R² ajustado si quieres una comparación rápida dentro de `statsmodels`,
y la validación cruzada si vas a tomar una decisión.

### A4 · Curvas: cuándo es razonable y cuándo es trampa

Un ajuste polinomial deja que la relación se doble. `PolynomialFeatures(degree=2)` añade el cuadrado de
la variable, el grado 3 añade el cubo, y así. Cada grado nuevo da más libertad para pasar cerca de los
puntos, y el R² de entrenamiento sube siempre —es la sección 3 otra vez—. La única forma de saber si la
curva es real es medirla sobre datos que el modelo no vio.

Dos relaciones de Comercial Andina, con los mismos cuatro grados:

1. **Clientes activos → ventas del mes por ciudad** (180 filas). Debería curvarse: los meses con muchos
   clientes son los meses buenos, y en los meses buenos el ticket también sube.
2. **Inversión en volantes → ventas mensuales nacionales** (30 filas). Aquí la muestra es diminuta.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

GRADOS = [1, 2, 3, 10]


def comparar_grados(X, y, etiqueta, grados=GRADOS):
    filas = []
    for g in grados:
        pipe = make_pipeline(StandardScaler(), PolynomialFeatures(g, include_bias=False),
                             LinearRegression())
        pipe.fit(X, y)
        filas.append({"caso": etiqueta, "grado": g, "R² entrenamiento": pipe.score(X, y),
                      "R² validación cruzada": cross_val_score(pipe, X, y, cv=kf,
                                                               scoring="r2").mean()})
    return pd.DataFrame(filas)


mk = marketing[marketing["ventas_mes"] > 0]
caso_a = comparar_grados(agg[["clientes"]], agg["ventas"], f"clientes → ventas ({len(agg)} filas)")
caso_b = comparar_grados(mk[["inversion_volantes"]], mk["ventas_mes"],
                         f"volantes → ventas ({len(mk)} filas)")
curvas = pd.concat([caso_a, caso_b], ignore_index=True)
curvas["diferencia"] = curvas["R² entrenamiento"] - curvas["R² validación cruzada"]
print(curvas.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (X, y, sub, titulo) in zip(axes, [
        (agg[["clientes"]], agg["ventas"], caso_a, "180 filas: el grado 2 es real"),
        (mk[["inversion_volantes"]], mk["ventas_mes"], caso_b, "30 filas: el grado 10 se inventa todo")]):
    ax.scatter(X.iloc[:, 0], y, s=18, alpha=0.55, color="#4C72B0")
    rej = np.linspace(X.iloc[:, 0].min(), X.iloc[:, 0].max(), 300).reshape(-1, 1)
    for g, color in zip(GRADOS, ["#B0B0B0", "#55A868", "#DD8452", "#C44E52"]):
        pipe = make_pipeline(StandardScaler(), PolynomialFeatures(g, include_bias=False),
                             LinearRegression()).fit(X, y)
        cv = sub.loc[sub["grado"] == g, "R² validación cruzada"].iloc[0]
        ax.plot(rej, pipe.predict(pd.DataFrame(rej, columns=X.columns)),
                color=color, linewidth=2 if g != 10 else 1.6,
                linestyle="--" if g == 10 else "-", label=f"grado {g} · VC {cv:.3f}")
    ax.set_ylim(y.min() - abs(y.min()) * 0.2, y.max() * 1.15)
    ax.set_xlabel(X.columns[0])
    ax.set_ylabel("ventas del mes")
    ax.set_title(titulo, fontsize=11)
    ax.legend(fontsize=8)
fig.suptitle("El grado 10 pasa más cerca de los puntos y predice peor en los dos casos",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

📌 **En el primer caso la curva es razonable; en el segundo es fraude.**

| caso | grado 1 | grado 2 | grado 3 | grado 10 |
|---|---|---|---|---|
| clientes → ventas (180 filas), R² validación | 0,5631 | **0,6416** | 0,6393 | 0,6004 |
| volantes → ventas (30 filas), R² validación | **0,7559** | 0,7493 | 0,7350 | **−0,1992** |

- **Grado 2 sobre 180 filas: adelante.** Gana casi 8 puntos de R² en validación cruzada (0,5631 →
  0,6416). La curvatura existe de verdad y el modelo la aprovecha.
- **Grado 3: sospechoso.** Baja a 0,6393. No aporta nada, así que no entra: entre dos modelos que
  predicen igual, se queda el más simple.
- **Grado 10 sobre 180 filas: sobreajuste claro.** El entrenamiento sube a 0,6614 y la validación cae a
  0,6004.
- **Grado 10 sobre 30 filas: trampa pura.** El R² de entrenamiento es 0,9426 —el más alto de toda la
  tabla, el número que alguien llevaría a una presentación— y el de validación cruzada es **−0,1992**.
  Negativo. Ese modelo predice peor que decir «la media de siempre». La curva se retuerce para tocar
  treinta puntos y, fuera de ellos, se dispara.

Regla operativa: **el grado se elige por validación cruzada, nunca mirando el gráfico.** Y si tienes
treinta observaciones, el grado 1 es casi siempre la respuesta correcta.

### A5 · El término de interacción

El diagnóstico de la sección 4 dice que al modelo de Advertising le falta un término, y la teoría del negocio dice cuál: la televisión rinde más donde ya hay radio. Eso es un **término de interacción**, el producto de las dos variables.

In [ ]:
adv_i = adv.copy()
adv_i["TVxRadio"] = adv_i["TV"] * adv_i["Radio"]
con_inter = sm.OLS(adv_i["Sales"],
                   sm.add_constant(adv_i[["TV", "Radio", "Newspaper", "TVxRadio"]])).fit()

print(con_inter.summary().tables[1])
comparativa = pd.DataFrame({
    "sin interacción": [multi.rsquared, multi.rsquared_adj,
                        linear_reset(multi, power=2, use_f=True).pvalue,
                        het_white(multi.resid, multi.model.exog)[1]],
    "con interacción": [con_inter.rsquared, con_inter.rsquared_adj,
                        linear_reset(con_inter, power=2, use_f=True).pvalue,
                        het_white(con_inter.resid, con_inter.model.exog)[1]],
}, index=["R²", "R² ajustado", "p-valor RESET (linealidad)", "p-valor White (homocedasticidad)"])
print()
print(comparativa.to_string(float_format=lambda v: f"{v:.4g}"))
print(f"\nEl R² sube de {multi.rsquared:.4f} a {con_inter.rsquared:.4f}, y el RESET sigue rechazando.")

⚠️ **El R² sube de 0,8972 a 0,9678 y los supuestos siguen sin cumplirse.** La interacción era real —su
coeficiente vale 0,0011 con t = 20,69— y arregla la mayor parte del problema, pero el RESET sigue
rechazando la linealidad. La conclusión honesta que va al informe es esta:

> *«El modelo con interacción explica el 96,8 % de la variación de las ventas y es el mejor de los que
> probamos. Las pruebas de diagnóstico indican que la forma funcional todavía no es exacta y que la
> varianza del error no es constante, así que los intervalos de predicción para un mercado individual
> no son fiables. Para decidir el reparto agregado de presupuesto, el modelo sirve.»*

Eso es lo que significa diagnosticar: **no es aprobar o suspender el modelo, es saber para qué se puede
usar y para qué no.**